In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import gc
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, f1_score
import mlflow.xgboost
import mlflow.lightgbm
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install mlflow

In [ ]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

import mlflow
mlflow.set_tracking_uri("file:///content/drive/MyDrive/ieee-fraud-detection/ml-flow-experiments")
mlflow.set_experiment("fraud_model_bakeoff")

<Experiment: artifact_location='file:///content/drive/MyDrive/ieee-fraud-detection/ml-flow-experiments/492684693097444956', creation_time=1784204345668, effective_trace_archival_retention=None, experiment_id='492684693097444956', last_update_time=1784204345668, lifecycle_stage='active', name='fraud_model_bakeoff', tags={}, trace_location=None, workspace='default'>

In [ ]:
drive_path = '/content/drive/MyDrive/ieee-fraud-detection/datasets'

files_in_dir = os.listdir(drive_path)
print(files_in_dir)

['sample_submission.csv', 'test_identity.csv', 'test_transaction.csv', 'train_identity.csv', 'train_transaction.csv', 'y_train.npy', 'y_test.npy', 'X_train_scaled.npy', 'X_test_scaled.npy', 'feature_names.json']


In [ ]:
X_train=np.load(drive_path+'/X_train_scaled.npy')
X_test=np.load(drive_path+'/X_test_scaled.npy')
y_train=np.load(drive_path+'/y_train.npy')
y_test=np.load(drive_path+'/y_test.npy')
features=json.load(open(drive_path+'/feature_names.json'))

In [ ]:
len(features)

432

In [ ]:
X_train.shape

(472432, 432)

In [ ]:
smote=SMOTE(random_state=42)
X_train_s,y_train_s=smote.fit_resample(X_train,y_train)

In [ ]:
print(y_train_s.sum())

455902


In [ ]:
xgb_param_dict={
    'n_estimators': randint(100,600),
    'max_depth': randint(3,10),
    'learning_rate': uniform(0.01,0.29),
    'subsample': uniform(0.6,0.4),
    'colsample_bytree': uniform(0.6,0.4),
    'min_child_weight': randint(1,10),
    'gamma': uniform(0,0.5),
    'reg_alpha': uniform(0,1),
    'reg_lambda': uniform(1,2),
    'scale_pos_weight': [1,3,5]
}

In [ ]:
xgb_search=RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric='auc',tree_method='hist'),
    param_distributions=xgb_param_dict,
    n_iter=10,
    scoring='roc_auc',
    cv=3,
    n_jobs=1,
    random_state=42,
    verbose=1
)

In [ ]:
gc.collect()

399

In [ ]:
xgb_search.fit(X_train_s,y_train_s)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=True,
                                           eval_metric='auc',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_constrain...
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7f32178fcfe0>,
                                        'reg_alpha': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7f32178fdf10>,
                                        'reg_lambda': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7f32178fd2b0>,
                                        'scale_pos_weight': [1, 3, 5],
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7f32178fe810>},
                   random_state=42, scoring='roc_auc', verbose=1)

In [ ]:
with mlflow.start_run(run_name="xgboost"):
      mlflow.log_params(xgb_search.best_params_)
      mlflow.log_metric("cv_auc", xgb_search.best_score_)
      best_model = xgb_search.best_estimator_
      test_proba = best_model.predict_proba(X_test)[:, 1]
      test_preds = best_model.predict(X_test)
      mlflow.log_metric("test_auc", roc_auc_score(y_test, test_proba))
      mlflow.log_metric("test_f1", f1_score(y_test, test_preds))
      mlflow.xgboost.log_model(best_model, name="model")

In [ ]:
runs = mlflow.search_runs(experiment_names=["fraud_model_bakeoff"])
print(runs[["run_id", "status", "tags.mlflow.runName", "metrics.test_auc"]])

                             run_id    status tags.mlflow.runName  \
0  cc984fe4e9fc462ca5d8cbdcff713108  FINISHED             xgboost   
1  62e118ef36fc46c3b50efaeb6580c1e7    FAILED             xgboost   
2  87eb08f81653420ba6e26c3389715407    FAILED             xgboost   
3  4c9100367c75455ab5252e19dab588ff    FAILED             xgboost   

   metrics.test_auc  
0          0.953905  
1          0.953905  
2          0.953905  
3               NaN  


In [ ]:
loaded_model=lgb.Booster(model_file='/content/lgbm_model.txt')
results=json.load(open('/content/lgbm_results.json'))

In [ ]:
results

{'best_params': {'colsample_bytree': 0.749816047538945,
  'learning_rate': 0.28570714885887566,
  'max_depth': 10,
  'min_child_samples': 65,
  'n_estimators': 120,
  'num_leaves': 117,
  'reg_alpha': 0.44583275285359114,
  'reg_lambda': 0.09997491581800289,
  'subsample': 0.7836995567863468},
 'cv_auc': 0.9967901373942848,
 'test_auc': 0.9612124625451384,
 'test_f1': 0.7235363690124187}

In [ ]:
with mlflow.start_run(run_name="lightgbm"):
  mlflow.log_params(results["best_params"])
  mlflow.log_metric("cv_auc",results["cv_auc"])
  mlflow.log_metric("test_auc",results["test_auc"])
  mlflow.log_metric("test_f1",results["test_f1"])
  mlflow.lightgbm.log_model(loaded_model,name="model")

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
runs = mlflow.search_runs(experiment_names=["fraud_model_bakeoff"])
print(runs[["run_id", "status", "tags.mlflow.runName", "metrics.test_auc", "metrics.test_f1", "metrics.cv_auc"]].to_string())

                             run_id    status tags.mlflow.runName  metrics.test_auc  metrics.test_f1  metrics.cv_auc
0  e6997eb70fb1466ea45d29a0d021fb51  FINISHED            lightgbm          0.961212         0.723536         0.99679
1  cc984fe4e9fc462ca5d8cbdcff713108  FINISHED             xgboost          0.953905         0.679606         0.99653
2  62e118ef36fc46c3b50efaeb6580c1e7    FAILED             xgboost          0.953905         0.679606         0.99653
3  87eb08f81653420ba6e26c3389715407    FAILED             xgboost          0.953905         0.679606         0.99653
4  4c9100367c75455ab5252e19dab588ff    FAILED             xgboost               NaN              NaN         0.99653


In [ ]:
lgbm_model=mlflow.lightgbm.load_model("runs:/e6997eb70fb1466ea45d29a0d021fb51/model")

In [ ]:
import shap

In [ ]:
explainer=shap.TreeExplainer(lgbm_model)
shap_values=explainer.shap_values(X_test)

shap_importance=pd.Series(
    (shap_values).mean(axis=0),
    index=features
).sort_values(ascending=False)

print(shap_importance.head(20))

amt_percent      0.079755
V130             0.062428
TransactionDT    0.014898
C9               0.014531
V30              0.008297
V90              0.008256
D3               0.007785
C8               0.007099
V246             0.006702
amt_log          0.006135
V64              0.005780
V203             0.005677
V299             0.004275
V48              0.004131
V250             0.003456
V313             0.003412
V304             0.003358
V228             0.002053
V220             0.002040
V98              0.001903
dtype: float64


In [ ]:
shap_importance = pd.Series(
    abs(shap_values).mean(axis=0),
    index=features
)

print(shap_importance.sum())
print((shap_importance < 0).sum())

12.422114944249191
0


In [ ]:
sorted_importance = shap_importance.sort_values(ascending=False)
cumulative = sorted_importance.cumsum() / sorted_importance.sum()

print(cumulative.iloc[49])
print(cumulative.iloc[99])
print(cumulative.iloc[149])
print(cumulative.iloc[199])

0.7129172679186127
0.8722171567553983
0.9357614946599493
0.968310588890923


In [ ]:
top_features=sorted_importance.index[:200].tolist()

features_idxs=[features.index(f) for f in top_features]


In [ ]:
print(features_idxs)

[422, 8, 426, 420, 14, 1, 27, 26, 344, 308, 6, 24, 423, 9, 329, 3, 149, 19, 4, 424, 421, 430, 427, 180, 7, 358, 12, 18, 178, 47, 31, 15, 46, 141, 404, 5, 362, 0, 429, 28, 120, 41, 428, 36, 147, 418, 145, 45, 333, 332, 345, 99, 22, 21, 29, 341, 11, 2, 390, 394, 30, 425, 335, 103, 406, 37, 395, 330, 25, 137, 44, 34, 98, 152, 391, 23, 13, 405, 363, 410, 331, 132, 251, 79, 153, 60, 239, 357, 86, 95, 49, 215, 62, 146, 403, 300, 419, 338, 181, 237, 337, 88, 360, 32, 176, 133, 206, 112, 199, 70, 431, 412, 202, 126, 69, 106, 365, 55, 128, 35, 401, 307, 409, 417, 63, 177, 16, 85, 305, 367, 140, 271, 113, 125, 80, 94, 370, 61, 134, 368, 350, 400, 39, 114, 104, 250, 364, 89, 97, 282, 197, 346, 174, 339, 43, 324, 343, 111, 186, 392, 50, 184, 17, 84, 349, 124, 117, 253, 396, 136, 74, 296, 261, 247, 293, 374, 309, 119, 127, 238, 226, 402, 116, 150, 151, 348, 229, 73, 105, 135, 252, 227, 220, 159, 356, 347, 148, 352, 312, 214]
